In [3]:
import numpy as np
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [4]:
# STEP 1: Imports, Setup, GPU Check

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np
import random
import matplotlib.pyplot as plt
import pickle
import csv
import copy
import os

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


# STEP 2: Shake-Shake Code (26, 2×64d)
# ~10-11M Params => More capacity => ~90%+ with enough training
def get_alpha_beta(batch_size, forward=True):
    """
    Shake-Shake random factors:
    - forward pass => alpha,beta ~ U(0,1)
    - backward pass => alpha,beta ~ 0.5
    """
    if forward:
        alpha = torch.rand(batch_size, 1, 1, 1, device=device)
        beta  = torch.rand(batch_size, 1, 1, 1, device=device)
    else:
        alpha = 0.5
        beta  = 0.5
    return alpha, beta

class ShakeFunction(torch.autograd.Function):
    """
    output = alpha*x1 + (1-alpha)*x2 in forward pass
    grad w.r.t x1 => beta, grad w.r.t x2 => (1-beta)
    """
    @staticmethod
    def forward(ctx, x1, x2):
        batch_size = x1.size(0)
        alpha, _ = get_alpha_beta(batch_size, forward=True)
        ctx.save_for_backward(alpha)
        return alpha * x1 + (1 - alpha) * x2

    @staticmethod
    def backward(ctx, grad_output):
        (alpha,) = ctx.saved_tensors
        batch_size = grad_output.size(0)
        _, beta = get_alpha_beta(batch_size, forward=False)
        grad_x1 = grad_output * beta
        grad_x2 = grad_output * (1 - beta)
        return grad_x1, grad_x2

def shake(x1, x2):
    return ShakeFunction.apply(x1, x2)

class ShakeBranch(nn.Module):
    """
    A single branch in a Shake-Shake block:
    conv->bn->relu->conv->bn
    """
    def __init__(self, in_ch, out_ch, stride):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_ch)
        )

    def forward(self, x):
        return self.net(x)

class ShakeShakeBlock(nn.Module):
    """
    2-branch block:
    output = skip + shake(branch1, branch2)
    """
    def __init__(self, in_ch, out_ch, stride):
        super().__init__()
        self.branch1 = ShakeBranch(in_ch, out_ch, stride)
        self.branch2 = ShakeBranch(in_ch, out_ch, stride)
        self.skip = None
        if stride != 1 or in_ch != out_ch:
            self.skip = nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False)

    def forward(self, x):
        if self.skip is not None:
            skip_out = self.skip(x)
        else:
            skip_out = x
        out1 = self.branch1(x)
        out2 = self.branch2(x)
        merged = shake(out1, out2)
        return F.relu(merged + skip_out, inplace=True)

def _make_shake_stage(n_blocks, in_ch, out_ch, stride):
    """
    Build a stage with n_blocks Shake-Shake blocks.
    """
    layers = []
    for i in range(n_blocks):
        s = stride if i == 0 else 1
        inc = in_ch if i == 0 else out_ch
        layers.append(ShakeShakeBlock(inc, out_ch, s))
    return nn.Sequential(*layers)

class ShakeShakeResNet_2x64d(nn.Module):
    """
    Shake-Shake (26, 2×64d) for CIFAR:
    - 3 stages
    - each stage has n=4 blocks => total layers = 3*(2n)+2=26
    - channel progression: 16 -> 64 -> 128 -> 256
    => ~10-11M parameters
    """
    def __init__(self, num_classes=10):
        super().__init__()
        self.in_ch = 16
        n = 4

        self.conv1 = nn.Conv2d(3, 16, 3, stride=1, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(16)

        self.stage1 = _make_shake_stage(n, in_ch=16,  out_ch=64, stride=1)
        self.stage2 = _make_shake_stage(n, in_ch=64,  out_ch=128, stride=2)
        self.stage3 = _make_shake_stage(n, in_ch=128, out_ch=256, stride=2)

        self.bn_final = nn.BatchNorm2d(256)
        self.relu = nn.ReLU(inplace=True)
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(256, num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = F.relu(out, inplace=True)

        out = self.stage1(out)
        out = self.stage2(out)
        out = self.stage3(out)

        out = self.bn_final(out)
        out = self.relu(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        return self.fc(out)

def create_shakeshake_2x64d(num_classes=10):
    return ShakeShakeResNet_2x64d(num_classes=num_classes)

# Instantiate & check param count
model = create_shakeshake_2x64d(num_classes=10).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model param count: {n_params:,} ~ ShakeShake (26, 2×64d) for higher accuracy")

# STEP 3: Data + Augmentations

MEAN = [0.4914, 0.4822, 0.4465]
STD  = [0.2023, 0.1994, 0.2010]

train_transform = transforms.Compose([
    transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.2),
    transforms.Normalize(MEAN, STD),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=train_transform
)
test_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=test_transform
)

BATCH_SIZE = 256
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# STEP 4: Label Smoothing + Mixup + CutMix

class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
        self.confidence = 1.0 - smoothing

    def forward(self, x, target):
        log_probs = F.log_softmax(x, dim=-1)
        with torch.no_grad():
            true_dist = torch.zeros_like(log_probs)
            true_dist.fill_(self.smoothing / (x.size(1) - 1))
            true_dist.scatter_(1, target.unsqueeze(1), self.confidence)
        return torch.mean(torch.sum(-true_dist * log_probs, dim=1))

def rand_bbox(size, lam):
    W = size[3]
    H = size[2]
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)
    return bbx1, bby1, bbx2, bby2

def dual_mix_data(x, y, alpha_mix=1.0, alpha_cut=1.0, p_cutmix=0.5):
    """
    Randomly use Mixup or CutMix.
    """
    if alpha_mix <= 0 and alpha_cut <= 0:
        return x, y, y, 1.0, "none"

    do_cutmix = (np.random.rand() < p_cutmix)
    if do_cutmix and alpha_cut > 0:
        lam = np.random.beta(alpha_cut, alpha_cut)
        batch_size = x.size(0)
        index = torch.randperm(batch_size).to(x.device)
        bbx1, bby1, bbx2, bby2 = rand_bbox(x.size(), lam)
        x[:, :, bby1:bby2, bbx1:bbx2] = x[index, :, bby1:bby2, bbx1:bbx2]
        lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (x.size(-1) * x.size(-2)))
        y_a, y_b = y, y[index]
        return x, y_a, y_b, lam, "cutmix"
    else:
        lam = np.random.beta(alpha_mix, alpha_mix)
        batch_size = x.size(0)
        index = torch.randperm(batch_size).to(x.device)
        mixed_x = lam * x + (1 - lam) * x[index, :]
        y_a, y_b = y, y[index]
        return mixed_x, y_a, y_b, lam, "mixup"

def dualmix_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# STEP 5: SWA Utilities

def bn_update(loader, model):
    """
    Update BN running_mean/var by one pass on data in train mode.
    """
    model.train()
    with torch.no_grad():
        for imgs, _ in loader:
            imgs = imgs.to(device)
            _ = model(imgs)

def update_swa_params(swa_model, model, alpha=1.0):
    """
    SWA param update:
    swa_param = alpha*swa_param + (1-alpha)*model_param
    """
    for swa_p, p in zip(swa_model.parameters(), model.parameters()):
        swa_p.data.mul_(alpha).add_(p.data, alpha=1 - alpha)


# STEP 6: Training & Evaluation

def train_one_epoch(
    model, loader, optimizer, criterion,
    alpha_mix=1.0, alpha_cut=1.0, p_cutmix=0.5
):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        inputs, y_a, y_b, lam, mode = dual_mix_data(
            imgs, labels,
            alpha_mix=alpha_mix, alpha_cut=alpha_cut,
            p_cutmix=p_cutmix
        )

        outputs = model(inputs)
        loss = dualmix_criterion(criterion, outputs, y_a, y_b, lam)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        _, pred = outputs.max(1)

        correct += pred.eq(y_a).sum().item()
        total += labels.size(0)

    return running_loss / total, 100.0 * correct / total

def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * imgs.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

    return running_loss / total, 100.0 * correct / total


# STEP 7: Main Training Loop (50 epochs)

criterion = LabelSmoothingCrossEntropy(smoothing=0.1)
optimizer = optim.SGD(
    model.parameters(),
    lr=0.1,
    momentum=0.9,
    weight_decay=5e-4,
    nesterov=True
)

EPOCHS = 50
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

SWA_START = 180
swa_model = copy.deepcopy(model)
swa_count = 0

best_acc_normal = 0.0

train_losses, train_accs = [], []
val_losses, val_accs = [], []

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion,
        alpha_mix=1.0, alpha_cut=1.0, p_cutmix=0.5
    )
    val_loss, val_acc = evaluate(model, test_loader, criterion)
    scheduler.step()

    if epoch >= SWA_START:

        swa_count += 1
        alpha = (swa_count - 1) / swa_count
        update_swa_params(swa_model, model, alpha=alpha)

    if val_acc > best_acc_normal:
        best_acc_normal = val_acc

    train_losses.append(train_loss)
    train_accs.append(train_acc)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    if (epoch+1) % 10 == 0 or epoch == EPOCHS-1:
        print(f"Epoch [{epoch+1:03d}/{EPOCHS}] "
              f"TrainLoss: {train_loss:.4f} | TrainAcc: {train_acc:.2f}% || "
              f"ValLoss: {val_loss:.4f} | ValAcc: {val_acc:.2f}% | "
              f"LR: {scheduler.get_last_lr()[0]:.4f}")

print("\nTraining complete!")
print(f"Best Test Accuracy (normal model) ~ {best_acc_normal:.2f}%")

if swa_count > 0:
    swa_model.to(device)

    bn_update(train_loader, swa_model)
    val_loss_swa, val_acc_swa = evaluate(swa_model, test_loader, criterion)
    print(f"SWA model test accuracy: {val_acc_swa:.2f}%")
    torch.save(swa_model.state_dict(), "shake_shake_2x64d_swa.pth")
else:
    torch.save(model.state_dict(), "shake_shake_2x64d_swa.pth")
    val_loss_swa, val_acc_swa = val_loss, val_acc

print(f"Final SWA Test Accuracy: {val_acc_swa:.2f}%")

# STEP 8: TTA + Submission CSV

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.title("Loss vs Epoch")
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.legend()

plt.subplot(1,2,2)
plt.title("Accuracy vs Epoch")
plt.plot(train_accs, label="Train Acc")
plt.plot(val_accs, label="Val Acc")
plt.legend()
plt.savefig("training_curves.png")
plt.close()


def tta_predict(model, img_np, n_augs=5):
    """
    For a single image, do 'n_augs' random augmentations, average predictions.
    """
    if img_np.shape == (3*32*32,):
        img_np = img_np.reshape(3,32,32).transpose(1,2,0)
    base_tensor = torch.from_numpy(img_np).float()/255.0
    base_tensor = base_tensor.permute(2,0,1)
    preds_sum = 0
    model.eval()
    with torch.no_grad():
        for _ in range(n_augs):
            pil_img = transforms.ToPILImage()(base_tensor)

            tta_tf = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize(MEAN, STD),
            ])
            aug_tensor = tta_tf(pil_img).unsqueeze(0).to(device)
            out = model(aug_tensor)
            preds_sum += F.softmax(out, dim=1)
        preds_mean = preds_sum / n_augs
        _, pred_label = preds_mean.max(1)
    return pred_label.item()


print("\nGenerating CSV for custom unlabeled data...\n")
custom_test_path = "/content/cifar_test_nolabel.pkl"
try:
    best_model = copy.deepcopy(swa_model)
    best_model.eval()

    with open(custom_test_path, "rb") as f:
        custom_test_data = pickle.load(f)
    unlabeled_imgs = custom_test_data[b'data']
    unlabeled_ids  = custom_test_data[b'ids']

    predictions = []
    for i in range(len(unlabeled_imgs)):
        label_pred = tta_predict(best_model, unlabeled_imgs[i], n_augs=5)
        predictions.append((unlabeled_ids[i], label_pred))

    csv_filename = "submission.csv"
    with open(csv_filename, "w", newline="") as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(["ID", "Labels"])
        for (img_id, label) in predictions:
            writer.writerow([img_id, label])

    print(f"TTA submission CSV saved as '{csv_filename}' with {len(predictions)} rows.")
except FileNotFoundError:
    print(f"Could not find '{custom_test_path}'. Make sure it's uploaded if you need a submission.")

Using device: cuda
Model param count: 11,653,594 ~ ShakeShake (26, 2×64d) for higher accuracy


100%|██████████| 170M/170M [00:13<00:00, 12.9MB/s]


Epoch [010/50] TrainLoss: 1.6465 | TrainAcc: 47.71% || ValLoss: 1.1290 | ValAcc: 75.06% | LR: 0.0905
Epoch [020/50] TrainLoss: 1.5174 | TrainAcc: 51.00% || ValLoss: 0.9417 | ValAcc: 84.54% | LR: 0.0655
Epoch [030/50] TrainLoss: 1.4318 | TrainAcc: 57.11% || ValLoss: 0.8889 | ValAcc: 87.53% | LR: 0.0345
Epoch [040/50] TrainLoss: 1.3607 | TrainAcc: 58.20% || ValLoss: 0.7944 | ValAcc: 92.21% | LR: 0.0095
Epoch [050/50] TrainLoss: 1.3115 | TrainAcc: 61.22% || ValLoss: 0.7694 | ValAcc: 93.54% | LR: 0.0000

Training complete!
Best Test Accuracy (normal model) ~ 93.64%
Final SWA Test Accuracy: 93.54%

Generating CSV for custom unlabeled data...

TTA submission CSV saved as 'submission.csv' with 10000 rows.
